Merging Stage

In [538]:
# merging tables
import pandas as pd
path = "../data/clean"
appointments = pd.read_csv(f'{path}/appointments_cleaned.csv')#ok
patients = pd.read_csv(f'{path}/patients_cleaned.csv')#ok
providers = pd.read_csv(f'{path}/providers_cleaned.csv')
payer = pd.read_csv(f'{path}/payers_cleaned.csv')
calendar = pd.read_csv(f'{path}/calendar_cleaned.csv')
facilities = pd.read_csv(f'{path}/facilities_cleaned.csv')#ok

In [539]:
merged_data = appointments.merge(
    patients,
    on='patient_id',
    how='left',
    validate='many_to_one'
)

In [540]:
merged_data = merged_data.merge(
    facilities,
    on='facility_id',
    how='left',
    validate='many_to_one'
)

In [541]:
merged_data = merged_data.merge(
    payer,
    on='payer_id',
    how='left',
    validate='many_to_one'
)

In [542]:
# check data info 
column_info = pd.DataFrame({
    'Column': merged_data.columns,
    'Data Type': merged_data.dtypes.values,
    'Non-Null Count': merged_data.notnull().sum().values,
    'Null Count': merged_data.isnull().sum().values,
    'Unique Values': merged_data.nunique().values
})

display(column_info)

,Column,Data Type,Non-Null Count,Null Count,Unique Values
0,appointment_id,str,36000,0,36000
1,patient_id,str,36000,0,10512
2,provider_id,str,36000,0,110
3,facility_id,str,36000,0,14
4,appointment_weekday,int64,36000,0,7
5,appointment_hour,int64,36000,0,13
6,lead_days,int64,36000,0,161
7,visit_type,str,36000,0,11
8,specialty_group,str,36000,0,8
9,payer_id,str,36000,0,8


In [543]:
missing_categorical_cols = [
    'sex_at_birth',
    'race_ethnicity_group',
    'preferred_language',
    'zip_area_id_y',
    'payer_primary_id',
    'pcp_provider_id'
]

merged_data[missing_categorical_cols] = merged_data[missing_categorical_cols].fillna('Unknown')

In [544]:
merged_data["chronic_condition_count"] = merged_data["chronic_condition_count"].fillna(
        merged_data["chronic_condition_count"].mode()[0]
    )    
merged_data["interpreter_needed_flag_y"] = merged_data["interpreter_needed_flag_y"].fillna(
        merged_data["interpreter_needed_flag_y"].mode()[0]
    )  
merged_data["sdoh_risk_score_y"] = merged_data["sdoh_risk_score_y"].fillna(
        merged_data["sdoh_risk_score_y"].median()
    )  

Handle data types

In [545]:
col_wrong_datatype = [
    "reminder_days_before",
    "wait_to_room_minutes",
    "interpreter_needed_flag_y",
    "chronic_condition_count"
]

merged_data[col_wrong_datatype] = merged_data[col_wrong_datatype].astype(int)
merged_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 36000 entries, 0 to 35999
Data columns (total 48 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   appointment_id               36000 non-null  str    
 1   patient_id                   36000 non-null  str    
 2   provider_id                  36000 non-null  str    
 3   facility_id                  36000 non-null  str    
 4   appointment_weekday          36000 non-null  int64  
 5   appointment_hour             36000 non-null  int64  
 6   lead_days                    36000 non-null  int64  
 7   visit_type                   36000 non-null  str    
 8   specialty_group              36000 non-null  str    
 9   payer_id                     36000 non-null  str    
 10  reminder_channel             36000 non-null  str    
 11  reminder_sent_flag           36000 non-null  int64  
 12  reminder_days_before         36000 non-null  int64  
 13  same_day_flag              

In [546]:
merged_data = merged_data.merge(
    calendar,
    left_on=['appointment_month', 'appointment_weekday'],
    right_on=['month_num', 'weekday_num'],
    how='left',
    validate='many_to_many'
)
merged_data = merged_data.drop(columns=['year','appointment_hour'])

In [547]:
# check data info 
column_info = pd.DataFrame({
    'Column': merged_data.columns,
    'Data Type': merged_data.dtypes.values,
    'Non-Null Count': merged_data.notnull().sum().values,
    'Null Count': merged_data.isnull().sum().values,
    'Unique Values': merged_data.nunique().values
})

display(column_info)

,Column,Data Type,Non-Null Count,Null Count,Unique Values
0,appointment_id,str,316054,0,36000
1,patient_id,str,316054,0,10512
2,provider_id,str,316054,0,110
3,facility_id,str,316054,0,14
4,appointment_weekday,int64,316054,0,7
5,lead_days,int64,316054,0,161
6,visit_type,str,316054,0,11
7,specialty_group,str,316054,0,8
8,payer_id,str,316054,0,8
9,reminder_channel,str,316054,0,4


Start Preprocessing Stage

In [548]:
categorical_cols = [
    'visit_type',
    'specialty_group',
    'reminder_channel',
    'sex_at_birth',
    'race_ethnicity_group',
    'preferred_language',
    'facility_type',
    'urban_rural_group',
    'payer_group',
    'service_line_focus'
]
numerical_cols = [
    'lead_days',
    'reminder_days_before',
    'distance_to_clinic_miles',
    'patient_age',
    'sdoh_risk_score_x',
    'sdoh_risk_score_y',
    'prior_completed_visits_12m',
    'prior_no_show_count_12m',
    'prior_cancel_count_12m',
    'chronic_condition_count'
]# deprivation_quintile==service_line_focus

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

target = merged_data['no_show_flag']
features = merged_data.drop(columns=[
    'no_show_flag', 'appointment_id', 'patient_id', 'provider_id',
    'facility_id', 'pcp_provider_id', 'zip_area_id_x', 'zip_area_id_y', 
    'zip_area_id', 'month_num', 'weekday_num'
])

x_train, x_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42, stratify=target
)

scaler = StandardScaler()
encoder = OneHotEncoder(sparse_output=False).set_output(transform="pandas")

train_encoded_df = encoder.fit_transform(x_train[categorical_cols])
test_encoded_df = encoder.transform(x_test[categorical_cols])

x_train[numerical_cols] = scaler.fit_transform(x_train[numerical_cols])
x_test[numerical_cols] = scaler.transform(x_test[numerical_cols])

x_train = x_train.drop(columns=categorical_cols)
x_test = x_test.drop(columns=categorical_cols)

x_train = x_train.join(train_encoded_df)
x_test = x_test.join(test_encoded_df)
